### Exploratory Data Analysis

In [2]:
import os
import json

import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
from scipy.stats import pearsonr, wilcoxon

In [42]:
# file locations 
ISLES_2024_TRAIN = "../../data/processed/"
ISLES_2024_TEST = "../../data/processed_test/"
ISLES_2024_TRAINR = "../../data/raw/derivatives/"
ISLES_2024_TESTR = "../../data/raw_test/derivatives/"
ISLES_2022_TRAIN = "../../data/isles_2022_train/"
ISLES_2022_TEST = "../../data/isles_2022_test/"

train_2022 = [f.path for f in os.scandir(ISLES_2022_TRAIN) if f.is_dir()]
test_2022 = [f.path for f in os.scandir(ISLES_2022_TEST) if f.is_dir()]

print(f"Num files in isles 2022 train: {len(train_2022)}")
print(f"Num files in isles 2022 test: {len(test_2022)}")

Num files in isles 2022 train: 174
Num files in isles 2022 test: 51


### Dataset analysis

In [43]:
raw_data_path = "/home/dayan/Documents/Datasets/stroke_segmentation/ISLES-2022"
dirs = os.listdir(raw_data_path)
dirs = sorted(dirs)
dirs = dirs[5:]


voxel_sizes = []
id = 1
for folder in dirs:
    path = f"{raw_data_path}/{folder}/ses-0001/dwi/{folder}_ses-0001_dwi.nii.gz"
    gt_path  = f"{raw_data_path}/derivatives/{folder}/ses-0001/{folder}_ses-0001_msk.nii.gz"
    dwi_image = nib.load(path)
    msk_image = nib.load(gt_path).get_fdata()
    voxel_volume = np.prod(dwi_image.header.get_zooms())/1000
    lesion_volume = np.sum(msk_image) * voxel_volume

    voxel_sizes.append({"Id":id, "lesion_vol": lesion_volume})
    id+=1
    
voxel_sizes = pd.DataFrame(voxel_sizes)
print(voxel_sizes.shape)
voxel_sizes.tail()

(250, 2)


,Id,lesion_vol
245,246,216.256010
246,247,18.256001
247,248,16.013596
248,249,11.251583
249,250,8.027989


I define the lesion categories as follows:
 - small: lesion volume < 5ml
 - medium: lesion volume >=5ml and < 20ml
 - large: lesion volume >= 20ml

In [44]:
def vol_count(arr):
    small_lesions = arr[(arr["lesion_vol"] < 5.0)]
    medium_lesions = arr[(arr["lesion_vol"] >= 5.0) & (arr["lesion_vol"] < 20.0)]
    large_lesions = arr[arr["lesion_vol"] >= 20]

    return small_lesions, medium_lesions, large_lesions

small, medium, large = vol_count(voxel_sizes)

print(f"small lesions: {small.shape[0]}")
print(f"medium lesions: {medium.shape[0]}")
print(f"large lesions: {large.shape[0]}")

small lesions: 104
medium lesions: 81
large lesions: 65


Check current train-test split volume distribution.

In [46]:
train = voxel_sizes.iloc[:174, :]
validation = voxel_sizes.iloc[175:199, :]
test = voxel_sizes.iloc[200:, :]

In [47]:
small, medium, large = vol_count(train)
print("Train")
smalls = small.shape[0]
mediums = medium.shape[0]
larges = large.shape[0]
print(f"small lesions: {smalls}")
print(f"medium lesions: {mediums}")
print(f"large lesions: {larges}")
print(f"Total samples: {smalls + mediums + larges}")

Train
small lesions: 91
medium lesions: 39
large lesions: 44
Total samples: 174


In [48]:
small, medium, large = vol_count(validation)

print("\nValidation")
smalls = small.shape[0]
mediums = medium.shape[0]
larges = large.shape[0]
print(f"small lesions: {smalls}")
print(f"medium lesions: {mediums}")
print(f"large lesions: {larges}")
print(f"Total samples: {smalls + mediums + larges}")


Validation
small lesions: 3
medium lesions: 18
large lesions: 3
Total samples: 24


In [49]:
small, medium, large = vol_count(test)

print("\nTest")
smalls = small.shape[0]
mediums = medium.shape[0]
larges = large.shape[0]
print(f"small lesions: {smalls}")
print(f"medium lesions: {mediums}")
print(f"large lesions: {larges}")
print(f"Total samples: {smalls + mediums + larges}")


Test
small lesions: 9
medium lesions: 23
large lesions: 18
Total samples: 50


### Results Analysis

In [53]:
def load_jsonl(file_path:str):

    data = []
    with open(file_path) as f:
        for line in f:
            data.append(json.loads(line))
        
    df = pd.DataFrame(data)
    return df


dual_single = "../../metrics/test/12_pre-trained_isles22_individual_metrics.jsonl"
single_single = "../../metrics/test/13_pre-trained_isles22_individual_metrics.jsonl"
unet = "../../metrics/test/14_pre-trained_isles22_individual_metrics.jsonl"
dual_dual_orig = "../../metrics/test/15_pre-trained_isles22_individual_metrics.jsonl"
dual_dual_corr = "../../metrics/test/16_pre-trained_isles22_individual_metrics.jsonl"

dual_single = load_jsonl(dual_single)
single_single = load_jsonl(single_single)
unet = load_jsonl(unet)
dual_dual_orig = load_jsonl(dual_dual_orig)
dual_dual_corr = load_jsonl(dual_dual_corr)

In [54]:
test_id = [i for i in range(200, 251)]
train_id = [i for i in range(1, 175) if i != 150 and i != 151 and i != 170]
val_id = [i for i in range(175, 200)]

dual_single["Id"] = test_id
single_single["Id"] = test_id
unet["Id"] = test_id
dual_dual_orig["Id"] = test_id
dual_dual_corr["Id"] = test_id

Verify Ids

In [12]:
with open('test_22.list', 'r') as file:
    test = [line.strip() for line in file]

with open('train_22.list', 'r') as file:
    train = [line.strip() for line in file]

print(f'There are {len(train)} file in the training set and {len(test)} in the testing set.')

There are 196 file in the training set and 51 in the testing set.


### Getting lesion volumes

In [36]:
no_lesion = voxel_sizes[voxel_sizes["lesion_vol"] == 0]
print(no_lesion.shape)
print(f"Id of no lesion samples:\n{no_lesion.Id.to_string(index=False)}")

have_lesion = voxel_sizes[voxel_sizes["lesion_vol"] > 0]
print(have_lesion.shape)

(3, 2)
Id of no lesion samples:
150
151
170
(247, 2)


3 samples in the whole dataset have no lesions while 247 samples have lesions. For the purposes of this analysis, the samples without lesions are removed. The Id numbers of the samples with no lesions are 150, 151 and 170 respectively.

In [16]:
gt_150 = raw_data_path + "/derivatives/sub-strokecase0150/ses-0001/sub-strokecase0150_ses-0001_msk.nii.gz"
gt_151 = raw_data_path + "/derivatives/sub-strokecase0151/ses-0001/sub-strokecase0151_ses-0001_msk.nii.gz"
gt_170 = raw_data_path + "/derivatives/sub-strokecase0170/ses-0001/sub-strokecase0170_ses-0001_msk.nii.gz"

loaded_150 = nib.load(gt_150).get_fdata()
loaded_151 = nib.load(gt_151).get_fdata()
loaded_170 = nib.load(gt_170).get_fdata()

print(np.unique(loaded_150))
print(np.unique(loaded_151))
print(np.unique(loaded_170))

[0.]
[0.]
[0.]


In [55]:
have_lesion_val = have_lesion[171:196]
have_lesion_test = have_lesion[196:]
print(have_lesion_val.head())

dual_single = pd.merge(left=dual_single, right=have_lesion_test, on="Id")
single_single = pd.merge(left=single_single, right=have_lesion_test, on="Id")
unet = pd.merge(left=unet, right=have_lesion_test, on="Id")
dual_dual_orig = pd.merge(left=dual_dual_orig, right=have_lesion_test, on="Id")
dual_dual_corr = pd.merge(left=dual_dual_corr, right=have_lesion_test, on="Id")

      Id  lesion_vol
174  175    2.527136
175  176   43.069075
176  177    0.280000
177  178   15.360001
178  179   28.416001


Size verification

Comparing the results of the single path bridger after training pipeline fixes and the UNet shows us that no significant difference exists between the two.

There is a significant difference between the performances in the training set so we analyse the dataset.

In [39]:
# metrics on lesion volume
val_median_vol = have_lesion_val["lesion_vol"].median()
val_mean_vol = have_lesion_val["lesion_vol"].mean()

test_median_vol = have_lesion_test["lesion_vol"].median()
test_mean_vol = have_lesion_test["lesion_vol"].mean()

print(f"Train lesion vol - median:{val_median_vol}, mean:{val_mean_vol}")
print(f"Test lesion vol - median:{test_median_vol}, mean:{test_mean_vol}")

Train lesion vol - median:10.88800051715225, mean:13.201910855638635
Test lesion vol - median:13.944000662304461, mean:35.38547076519415


In [40]:
val_small_lesion_count = have_lesion_val[have_lesion_val["lesion_vol"] < 5].shape[0]
test_small_lesion_count = have_lesion_test[have_lesion_test["lesion_vol"] < 5].shape[0]

print(f"Proportion of small lesions val: {val_small_lesion_count/have_lesion_val.shape[0]}")
print(f"Proportion of small lesions test: {test_small_lesion_count/have_lesion_test.shape[0]}")

Proportion of small lesions val: 0.16
Proportion of small lesions test: 0.17647058823529413


### Test of Isles'22 F1 score calculation 

In [56]:
thresholds = [
    ('very small', 0, 5),
    ('medium',     5, 15),
    ('large',      15, 9999)
]

for label, low, high in thresholds:
    for name, df in [('Dual path single bridger', dual_single),
                     ('Single path no bridger',       single_single),
                     ('unet', unet),
                     ('Dual path original bridger', dual_dual_orig),
                     ('Dual path corrected bridger', dual_dual_corr)]:
        sub = df[(df['lesion_vol'] >= low) & 
                 (df['lesion_vol'] <  high)]
        if len(sub) > 0:
            print(f"{label} ({low}-{high}ml) {name} "
                  f"n={len(sub):2d}  "
                  f"Dice={sub['dice'].mean():.3f}  "
                  f"F1={sub['F1'].mean():.3f}")
    print()

very small (0-5ml) Dual path single bridger n= 9  Dice=0.496  F1=0.480
very small (0-5ml) Single path no bridger n= 9  Dice=0.454  F1=0.453
very small (0-5ml) unet n= 9  Dice=0.480  F1=0.476
very small (0-5ml) Dual path original bridger n= 9  Dice=0.501  F1=0.461
very small (0-5ml) Dual path corrected bridger n= 9  Dice=0.498  F1=0.442

medium (5-15ml) Dual path single bridger n=19  Dice=0.738  F1=0.542
medium (5-15ml) Single path no bridger n=19  Dice=0.723  F1=0.482
medium (5-15ml) unet n=19  Dice=0.696  F1=0.430
medium (5-15ml) Dual path original bridger n=19  Dice=0.751  F1=0.509
medium (5-15ml) Dual path corrected bridger n=19  Dice=0.749  F1=0.504

large (15-9999ml) Dual path single bridger n=23  Dice=0.802  F1=0.522
large (15-9999ml) Single path no bridger n=23  Dice=0.804  F1=0.521
large (15-9999ml) unet n=23  Dice=0.787  F1=0.490
large (15-9999ml) Dual path original bridger n=23  Dice=0.805  F1=0.527
large (15-9999ml) Dual path corrected bridger n=23  Dice=0.805  F1=0.500

